In [ ]:
!pip install -U transformers

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_datasets = load_dataset("glue", "mrpc")

In [ ]:
def tokenize_function(example):
  return tokenizer(example["sentence1"], example["sentence2"], truncation = True)


tokenizer_dataset = raw_datasets.map(tokenize_function, batched = True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
!pip install evaluate

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import numpy as np

In [ ]:
def compute_metrics(eval_preds):
  metric = evaluate.load("glue", "mrpc")
  logits, labels = eval_preds
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch")

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenizer_dataset["train"],
    eval_dataset=tokenizer_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# default epoch of  TrainingArguments =3
trainer.train()

advanced tranied feature

In [ ]:
# mixed precision training: fpt16 = True Use fp16=True in your training arguments for faster training and reduced memory usage
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    fp16=True,  # Enable mixed precision
)
# Gradient Accumulation: For effective larger batch sizes when GPU memory is limited

training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 4 * 4 = 16
)
# Learning Rate Scheduling: The Trainer uses linear decay by default
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",  # Try different schedulers
)



# Full training loop

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_scheduler
from datasets import load_dataset
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(device)
device

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
raw_data = load_dataset("glue", "mrpc")
tokenizer_dataset = raw_data.map(tokenize_function, batched = True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer )


In [ ]:
tokenizer_dataset = tokenizer_dataset.remove_columns(["sentence1", "sentence2", "idx"])
tokenizer_dataset = tokenizer_dataset.rename_column("label", "labels")
tokenizer_dataset.set_format("torch")

tokenizer_dataset["train"].column_names

In [ ]:
train_Dataloader = DataLoader(tokenizer_dataset["train"], batch_size=8, shuffle= True, collate_fn= data_collator)
val_Dataloader = DataLoader(tokenizer_dataset["validation"], batch_size=8, collate_fn= data_collator)

In [ ]:
for batch in train_Dataloader:
  break
{k:v.shape for k, v in batch.items()}

In [ ]:
num_epochs = 3
optimizer = AdamW(model.parameters(), lr =5e-5 )
num_training_steps = num_epochs * len(train_Dataloader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

Train

In [ ]:
from tqdm.auto import tqdm
model.train()
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
  for batch in train_Dataloader:
    batch = {k:v.to(device) for k,v in batch.items()}
    output = model(**batch)
    loss = output.loss
    loss.backward()

    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)





evaluation

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in val_Dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

training with Accelerate ( multiple GPUs or TPUs)

In [ ]:
from accelerate import Accelerator
accelerator = Accelerator()
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_Dataloader, val_Dataloader, model, optimizer
)

num_epochs = 1
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)